In [ ]:
import os
import PIL as pil
from tqdm import tqdm
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as F
import cv2
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
IMG_SIZE = 512
IMG_DIR = 'data/'
BATCH_SIZE = 4
NUM_WORKERS = 4

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Assuming that we are on a CUDA machine, this should print a CUDA device:

print(device)

In [ ]:
def get_folder_structure(base = 'data'):
    """
    Returns the folder structure of the data folder
    """
    return [x[0] for x in os.walk(base)]
get_folder_structure()

In [ ]:
class BenGrahamPreprocessing:
    def __init__(self, img_size):
        self.img_size = img_size

    def __call__(self, image):
       
        image_np = image.permute(1, 2, 0).numpy()  # Cambiar a (H, W, C) para OpenCV

        # Aplicar el desenfoque gaussiano de OpenCV
        blurred_image = cv2.GaussianBlur(image_np, (0, 0), self.img_size/10)

        # Combinar la imagen original y la desenfocada
        combined_image = cv2.addWeighted(image_np, 4, blurred_image, -4, 128)

        # Convertir de regreso a Tensor
        combined_image_tensor = transforms.ToTensor()(combined_image)

        return combined_image_tensor


In [ ]:
transform = transforms.Compose(
    [transforms.Resize((IMG_SIZE, IMG_SIZE)),
    #  transforms.Grayscale(num_output_channels=1),
     transforms.ToTensor(),
     BenGrahamPreprocessing(IMG_SIZE),
     transforms.Normalize(mean=0.5, std=0.5)])

In [ ]:
# Load the training data from the data folder
trainset = torchvision.datasets.ImageFolder(root='data/train/', transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE,
                                          shuffle=True, num_workers=NUM_WORKERS)
testset = torchvision.datasets.ImageFolder(root='data/test/', transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE,
                                         shuffle=False, num_workers=NUM_WORKERS)

classes = ('0-noDR', '1-mild', '2-moderate', '3-severe', '4-proliferativeDR')

In [ ]:

data_iter = iter(trainloader)
images, labels = next(data_iter)

# Mostrar algunas imágenes del lote procesado
plt.figure(figsize=(10, 10))
for i in range(BATCH_SIZE):
    ax = plt.subplot(3, 2, i + 1)
    plt.imshow(images[i].permute(1, 2, 0))  # Cambiar el orden de los canales para visualización
    plt.title(f'Label: {labels[i]}')
    plt.axis('off')
plt.show()

In [ ]:
# Cargar un modelo preentrenado
model = torchvision.models.resnet50(pretrained=True)

# Ajustar la capa final para tu número de clases (puedes cambiar este ejemplo si hay más de 2 clases)
num_classes = 5  # Cambia a tu número de clases si es diferente
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

# Llevar el modelo a GPU si está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
# criterion = torch.nn.NLLLoss(reduction="sum")  # Usa NLLLoss para clasificación binaria
criterion = torch.nn.CrossEntropyLoss()  # Usa CrossEntropy para clasificación múltiple
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',factor=0.5, patience=20,verbose=1)

In [ ]:
num_epochs = 10  # Ajusta como sea necesario

for epoch in range(num_epochs):
    # model.train()  # Establece el modo de entrenamiento
    running_loss = 0.0

    for images, labels in tqdm(trainloader):
        images, labels = images.to(device), labels.to(device)  # Mueve a GPU si es necesario

        optimizer.zero_grad()  # Limpia los gradientes

        outputs = model(images)  # Propagación hacia adelante

        loss = criterion(outputs, labels)  # Calcular la pérdida
        loss.backward()  # Propagación hacia atrás
        optimizer.step()  # Actualizar pesos

        running_loss += loss.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(trainloader):.4f}')

In [ ]:
model.eval()  # Establece modo de evaluación
correct = 0
total = 0

with torch.no_grad():  # Desactiva el cálculo de gradientes para evaluación
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Precisión en el conjunto de prueba: {100 * correct / total:.2f}%')